In [2]:
# make sure that the needed packages are installed beforehand! if need be, uncomment and run:
# !pip install numpy pandas scipy mat73

import numpy as np
import pandas as pd
import scipy.io as sio
import mat73

def load_mat_any(path):
    try:
        mat = mat73.loadmat(path)
    except TypeError:
        # fall back to scipy in case
        mat = sio.loadmat(path, simplify_cells=True)
    return mat

In [5]:
CSV_PATH = "M:/Journal Publications/_Submitted/2025_Nwokeabia_MiRPNI/Submission 3/dataset_v4/test/P1_S1_EMG1kHz.csv"
META_PATH = "M:/Journal Publications/_Submitted/2025_Nwokeabia_MiRPNI/Submission 3/dataset_v4/test/P1_S1_meta.json"
CHANNELS_PATH = "M:/Journal Publications/_Submitted/2025_Nwokeabia_MiRPNI/Submission 3/dataset_v4/test/P1_metadata.json"
TASKS_PATH = "M:/Journal Publications/_Submitted/2025_Nwokeabia_MiRPNI/Submission 3/dataset_v4/test/movements.json"

# 1. Trial-level metadata (already one row per trial)
trial_meta = pd.read_json(META_PATH)

# 2. Reshape EMG csv: group by TrialID -> (numSamples, numChannels) array per trial
emg_raw = pd.read_csv(CSV_PATH)
channel_cols = [c for c in emg_raw.columns if c.startswith('EMG1k_')]

emg_arrays = {
    tid: group[channel_cols].to_numpy()
    for tid, group in emg_raw.groupby('TrialID')
}
trial_meta['EMG1k'] = trial_meta['TrialID'].map(emg_arrays)

# sanity check
print(trial_meta['EMG1k'].iloc[0].shape)  # (8000, 8) confirmed

# 3. Task names
tasks = pd.read_json(TASKS_PATH)
tasks = tasks.rename(columns={'movementNumber': 'TaskNumber', 'movementName': 'TaskName'})
tasks['TaskNumber'] = tasks['TaskNumber'].astype(int)
trial_meta['TaskNumber'] = trial_meta['TaskNumber'].astype(int)
trial_meta = trial_meta.merge(tasks[['TaskNumber', 'TaskName']], on='TaskNumber', how='left')

# 4. Channel names -- lookup dict, not merged row-wise (constant across all trials)
channels = pd.read_json(CHANNELS_PATH)
channel_names = channels.set_index('channelNumber')['channelName'].to_dict()
# {1: 'FDPI', 2: 'FCR', 3: 'Ulnar RPNI', 4: 'Median RPNI', 5: 'EDC', 6: 'EPL', 7: 'FDPS', 8: 'FPL'}

trial_meta.head()

(8000, 8)


,TrialID,TaskNumber,TrialNumber,RestTime,HoldTime,EMG1k,TaskName
0,1,1,1,5000,3000,"[[-6.25, -5.0, 11.25, 19.25, -4.25, -8.5, -1.2...",rest
1,2,1,2,5000,3000,"[[-12.5, -1.25, -0.25, 19.0, -9.5, -10.5, 1.0,...",rest
2,3,1,3,5000,3000,"[[-3.5, 3.25, 19.0, 15.25, -2.5, -16.75, -5.0,...",rest
3,4,1,4,5000,3000,"[[3.5, -7.75, -2.0, 17.75, -18.75, -13.0, -5.0...",rest
4,5,1,5,5000,3000,"[[-10.25, 2.25, 4.75, 22.75, -2.0, -4.25, -1.7...",rest
